# Accessing Vortex Profile Matching Data from OPeNDAP, S3, or Local Storage

This tutorial shows how to access the Vortex Profile Matching dataset from OPeNDAP, S3, or local storage downloaded from Zenodo.
While I strongly recommend using pixi to create and manage your Python environments, I am also providing the version of icechunk and xarray used to create this notebook for reference. The `pixi.toml` and `pixi.lock` files are available in the repository and we are using the `default` environment.

In [3]:
import xarray as xr
import icechunk

print("xarray version:", xr.__version__)
print("icechunk version:", icechunk.__version__)

xarray version: 2025.8.0
icechunk version: 1.1.4


Defining the domain, dataset and variable to access. Here we are simply accessing all natcged temperature CTD profiles.

In [4]:
domain = "matched"
dataset = "ctd"
variable = "Temperature"

## Local Access downloaded from Zenodo

Here we access the data via local storage using icechunk combined with xarray. Let's assume that you have downloaded and unzipped the data from Zenodo into a local directory called `../data`.

In [ ]:
base_path = "../data"
getpath = lambda domain,dataset,variable: f"{base_path}/{domain.title()}/{dataset.lower()}_{variable.title()}"


storage = icechunk.local_filesystem_storage(getpath(domain, dataset, variable))
session = icechunk.Repository.open(storage).readonly_session(branch="main")
temp = xr.open_zarr(session.store, consolidated=False)

# if you want, you can export to netcdf
# temp.to_netcdf("ctd_temperature.nc")

temp

<xarray.Dataset> Size: 2GB
Dimensions:                             (casts: 31917, NbSample: 20,
                                         levels: 8419)
Coordinates:
  * casts                               (casts) int32 128kB 19423233 ... 1382...
    lon                                 (casts) float32 128kB dask.array<chunksize=(1000,), meta=np.ndarray>
    lat                                 (casts) float32 128kB dask.array<chunksize=(1000,), meta=np.ndarray>
  * levels                              (levels) float32 34kB 0.0 ... 8.418e+03
    time                                (casts) datetime64[ns] 255kB dask.array<chunksize=(1000,), meta=np.ndarray>
    z                                   (casts, levels) float32 1GB dask.array<chunksize=(1000, 8419), meta=np.ndarray>
Dimensions without coordinates: NbSample
Data variables: (12/35)
    eddy_amplitude                      (casts) float32 128kB dask.array<chunksize=(1000,), meta=np.ndarray>
    eddy_Ro                             (casts) float32 128kB dask.array<chunksize=(1000,), meta=np.ndarray>
    eddy_effective_contour_latitude     (casts, NbSample) float32 3MB dask.array<chunksize=(1000, 20), meta=np.ndarray>
    eddy_effective_area                 (casts) float32 128kB dask.array<chunksize=(1000,), meta=np.ndarray>
    eddy_effective_contour_height       (casts) float32 128kB dask.array<chunksize=(1000,), meta=np.ndarray>
    eddy_effective_contour_shape_error  (casts) float32 128kB dask.array<chunksize=(1000,), meta=np.ndarray>
    ...                                  ...
    platform                            (casts) |S80 3MB dask.array<chunksize=(1000,), meta=np.ndarray>
    wod_cruise_identifier               (casts) |S8 255kB dask.array<chunksize=(1000,), meta=np.ndarray>
    eddy_uavg_profile                   (casts, NbSample) float32 3MB dask.array<chunksize=(1000, 20), meta=np.ndarray>
    temperature                         (casts, levels) float32 1GB dask.array<chunksize=(1000, 8419), meta=np.ndarray>
    eddy_umax                           (casts) float32 128kB dask.array<chunksize=(1000,), meta=np.ndarray>
    project                             (casts) |S80 3MB dask.array<chunksize=(1000,), meta=np.ndarray>
Attributes: (12/38)
    institution:                   National Centers for Environmental Informa...
    source:                        World Ocean Database
    references:                    World Ocean Database 2018. URL:http://data...
    title:                         World Ocean Database - Multi-cast file
    summary:                       Data for multiple casts from the World Oce...
    id:                            /nodc/data/OCLAB/OCL/retrieval/monthlies/w...
    ...                            ...
    history:                       
    license:                       
    standard_name_vocabulary:      CF Standard Name Table v41
    featureType:                   Profile
    cdm_data_type:                 Profile
    Conventions:                   CF-1.6

## OPeNDAP Access

Here we access the data via OPeNDAP using xarray's built-in functionality.

In [3]:
base_url = f"http://www.smast.umassd.edu:8081/thredds/dodsC/Vortex_profiles/vortex_profiles"
# for WHOI server
# base_url = f"http://bigcreek.whoi.edu:8080/thredds/dodsC/VORTEX_PROFILE"

geturl = lambda domain,dataset,variable: f"{base_url}/{domain.title()}/{domain}_{dataset.lower()}_{variable.title()}.nc"

temp = xr.open_dataset(geturl(domain,dataset,variable.lower()))

temp

<xarray.Dataset> Size: 2GB
Dimensions:                             (casts: 31917, NbSample: 20,
                                         levels: 8419)
Coordinates:
  * levels                              (levels) float32 34kB 0.0 ... 8.418e+03
  * casts                               (casts) int32 128kB 19423233 ... 1382...
    lat                                 (casts) float32 128kB ...
    lon                                 (casts) float32 128kB ...
    time                                (casts) datetime64[ns] 255kB ...
    z                                   (casts, levels) float32 1GB ...
Dimensions without coordinates: NbSample
Data variables: (12/35)
    project                             (casts) |S64 2MB ...
    platform                            (casts) |S64 2MB ...
    wod_cruise_identifier               (casts) |S64 2MB ...
    eddy_latitude_max                   (casts) float32 128kB ...
    eddy_effective_area                 (casts) float32 128kB ...
    eddy_speed_contour_shape_error      (casts) float32 128kB ...
    ...                                  ...
    eddy_cost_association               (casts) float32 128kB ...
    eddy_num_point_s                    (casts) float32 128kB ...
    eddy_f                              (casts) float32 128kB ...
    eddy_longitude                      (casts) float32 128kB ...
    eddy_speed_contour_longitude        (casts, NbSample) float32 3MB ...
    temperature                         (casts, levels) float32 1GB ...
Attributes: (12/40)
    institution:                   National Centers for Environmental Informa...
    source:                        World Ocean Database
    references:                    World Ocean Database 2018. URL:http://data...
    title:                         World Ocean Database - Multi-cast file
    summary:                       Data for multiple casts from the World Oce...
    id:                            /nodc/data/OCLAB/OCL/retrieval/monthlies/w...
    ...                            ...
    standard_name_vocabulary:      CF Standard Name Table v41
    featureType:                   Profile
    cdm_data_type:                 Profile
    Conventions:                   CF-1.6
    DODS.strlen:                   8
    DODS.dimName:                  string8

# AWS S3 Access (Icechunk)

Here we access the data via AWS S3 using the icechunk library combined with xarray.

In [7]:
storage = icechunk.s3_storage(
    bucket="iuryt-shared",
    prefix=f"icechunk/ocean/vortex-profiles/{domain.title()}/{dataset}_{variable}",
    region="us-west-2",
    anonymous=True,
)

repo = icechunk.Repository.open(storage)
session = repo.readonly_session("main")

temp = xr.open_zarr(session.store, consolidated=False)
temp

<xarray.Dataset> Size: 2GB
Dimensions:                             (casts: 31917, NbSample: 20,
                                         levels: 8419)
Coordinates:
  * casts                               (casts) int32 128kB 19423233 ... 1382...
    lon                                 (casts) float32 128kB dask.array<chunksize=(1000,), meta=np.ndarray>
  * levels                              (levels) float32 34kB 0.0 ... 8.418e+03
    time                                (casts) datetime64[ns] 255kB dask.array<chunksize=(1000,), meta=np.ndarray>
    lat                                 (casts) float32 128kB dask.array<chunksize=(1000,), meta=np.ndarray>
    z                                   (casts, levels) float32 1GB dask.array<chunksize=(1000, 8419), meta=np.ndarray>
Dimensions without coordinates: NbSample
Data variables: (12/35)
    eddy_cost_association               (casts) float32 128kB dask.array<chunksize=(1000,), meta=np.ndarray>
    eddy_Ro                             (casts) float32 128kB dask.array<chunksize=(1000,), meta=np.ndarray>
    eddy_effective_contour_height       (casts) float32 128kB dask.array<chunksize=(1000,), meta=np.ndarray>
    eddy_effective_contour_longitude    (casts, NbSample) float32 3MB dask.array<chunksize=(1000, 20), meta=np.ndarray>
    eddy_effective_contour_latitude     (casts, NbSample) float32 3MB dask.array<chunksize=(1000, 20), meta=np.ndarray>
    eddy_amplitude                      (casts) float32 128kB dask.array<chunksize=(1000,), meta=np.ndarray>
    ...                                  ...
    platform                            (casts) |S80 3MB dask.array<chunksize=(1000,), meta=np.ndarray>
    eddy_uavg_profile                   (casts, NbSample) float32 3MB dask.array<chunksize=(1000, 20), meta=np.ndarray>
    project                             (casts) |S80 3MB dask.array<chunksize=(1000,), meta=np.ndarray>
    wod_cruise_identifier               (casts) |S8 255kB dask.array<chunksize=(1000,), meta=np.ndarray>
    eddy_umax                           (casts) float32 128kB dask.array<chunksize=(1000,), meta=np.ndarray>
    temperature                         (casts, levels) float32 1GB dask.array<chunksize=(1000, 8419), meta=np.ndarray>
Attributes: (12/38)
    institution:                   National Centers for Environmental Informa...
    source:                        World Ocean Database
    references:                    World Ocean Database 2018. URL:http://data...
    title:                         World Ocean Database - Multi-cast file
    summary:                       Data for multiple casts from the World Oce...
    id:                            /nodc/data/OCLAB/OCL/retrieval/monthlies/w...
    ...                            ...
    history:                       
    license:                       
    standard_name_vocabulary:      CF Standard Name Table v41
    featureType:                   Profile
    cdm_data_type:                 Profile
    Conventions:                   CF-1.6